# Synchronizing MCP server capabilities with AgentCore Gateway

## Overview

Tool, prompt, and resource definitions on an MCP server change over time. AgentCore Gateway has three mechanisms for keeping its catalog in sync with what each MCP server target actually exposes:

1. **Explicit synchronization** — call `SynchronizeGatewayTargets` on demand after the upstream MCP server changes.
2. **Implicit synchronization** — `CreateGatewayTarget` and `UpdateGatewayTarget` always re-read the upstream server's catalog as part of the operation.
3. **Dynamic listing** (`listingMode='DYNAMIC'`) — Gateway forwards every list request (`tools/list`, `prompts/list`, `resources/list`, `resources/templates/list`) to the MCP server, so no synchronization is ever required.

> **How (1) and (2) relate to (3).** Explicit and implicit synchronization are both **control-plane operations on `listingMode='DEFAULT'` targets** — i.e. they populate the AgentCore Gateway *cache* that DEFAULT-mode list calls will be answered from. `CreateGatewayTarget` is the very first cache fill (implicit at create time); `UpdateGatewayTarget` refills it as a side effect of every update; `SynchronizeGatewayTargets` is the on-demand refill in between. DYNAMIC-mode targets are not cached during create or update operations. 

## Workshop roadmap

| Step | What you do |
|---|---|
| **1** | Set up the notebook environment (env vars, utilities, logging). |
| **2** | Create the AgentCore Gateway: Cognito inbound auth, IAM role, then the Gateway itself. |
| **3** | Deploy the initial FastMCP server (just `getOrder` + `updateOrder` for now) to AgentCore Runtime. |
| **4** | Wire that MCP Server in as a Gateway target (outbound OAuth, target creation, inbound token, `GatewayMCPClient` helper). |
| **5** | Demonstrate **explicit synchronization** — add a tool, redeploy, observe the gateway catalog stays stale until `SynchronizeGatewayTargets` runs. |
| **6** | Demonstrate **implicit synchronization** — add another tool, redeploy, then call `UpdateGatewayTarget` and watch the catalog refresh as a side effect. |
| **7** | Demonstrate **dynamic listing** — create a second target with `listingMode='DYNAMIC'`, and compare cached vs live across list tools operations. |
| **8** | Clean up. |

## Tutorial Details

| Information          | Details                                                              |
|:---------------------|:---------------------------------------------------------------------|
| Tutorial type        | Interactive                                                          |
| AgentCore components | AgentCore Gateway, AgentCore Identity, AgentCore Runtime             |
| Agentic Framework    | Strands Agents                                                       |
| Gateway Target type  | MCP server                                                           |
| MCP primitives       | Tools, Prompts, Resources (static and templated)                     |
| Inbound Auth IdP     | Amazon Cognito, but can use others                                   |
| Outbound Auth        | Amazon Cognito, but can use others                                   |
| LLM model            | Anthropic Claude Haiku 4.5                                           |
| Tutorial components  | Explicit sync, implicit sync, dynamic listing                        |
| Tutorial vertical    | Cross-vertical                                                       |
| Example complexity   | Easy                                                                 |
| SDK used             | boto3                                                                |

### Step 1: Setup & Prerequisites

To execute this tutorial you will need:
* Jupyter notebook (Python kernel)
* uv
* AWS credentials
* Amazon Cognito
* The gateway, runtime, OAuth2 credential provider, and target created in `01-mcp-server-target.ipynb`

In [ ]:
# Install from the requirements file or pyproject.toml file in current directory
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using Amazon SageMaker notebook
import os
from time import sleep

# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")

In [ ]:
# Import utils
import os
import sys

# Get the directory of the current script
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

# Setup logging
import logging

# Configure logging for notebook environment
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

# Set specific logger levels
logging.getLogger("gateway").setLevel(logging.INFO)

## Step 2: Create AgentCore Gateway

### Step 2.1: Cognito user pool for gateway inbound auth

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {
        "ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
        "ScopeDescription": "Scope for invoking the agentcore gateway",
    },
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
scopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
gw_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {gw_user_pool_id}")

utils.get_or_create_resource_server(
    cognito, gw_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(
    cognito, gw_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)

print(f"Client ID: {gw_client_id}")

# Get discovery URL
gw_cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{gw_user_pool_id}/.well-known/openid-configuration"
print(gw_cognito_discovery_url)

### Step 2.2: Create AgentCore Gateway IAM Role

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-mcpgateway")
print("AgentCore Gateway role ARN:", agentcore_gateway_iam_role["Role"]["Arn"])

### Step 2.3: Create AgentCore Gateway

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [gw_client_id],
        "discoveryUrl": gw_cognito_discovery_url,
    }
}

# IMPORTANT: searchType="NONE" — Step 7's DYNAMIC target requires this.
# Per gateway-target-MCPservers.html: "Dynamic MCP targets
# (listingMode=DYNAMIC) are not supported on gateways with semantic search
# enabled." Notebook 01 uses searchType="SEMANTIC" because it doesn't create
# DYNAMIC targets.
create_response = gateway_client.create_gateway(
    name="ac-gateway-mcp-server",
    roleArn=agentcore_gateway_iam_role["Role"]["Arn"],
    protocolType="MCP",
    protocolConfiguration={"mcp": {"supportedVersions": ["2025-11-25"]}},
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with MCP Server target (sync demos)",
)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID:  {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

## Step 3: Deploy the initial MCP server on AgentCore Runtime

### Step 3.1: Cognito user pool for runtime inbound auth

This pool is for the Cognito-based JWT authorizer the AgentCore Runtime expects on inbound calls — separate from the gateway's own inbound pool above.

In [ ]:
# Creating Cognito User Pool
import os
import boto3

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-runtime-pool"
RESOURCE_SERVER_ID = "sample-agentcore-runtime-id"
RESOURCE_SERVER_NAME = "sample-agentcore-runtime-name"
CLIENT_NAME = "sample-agentcore-runtime-client"
SCOPES = [
    {
        "ScopeName": "invoke",  # Just 'invoke', will be formatted as resource_server_id/invoke
        "ScopeDescription": "Scope for invoking the agentcore gateway",
    },
]

scope_names = [f"{RESOURCE_SERVER_ID}/{scope['ScopeName']}" for scope in SCOPES]
runtimeScopeString = " ".join(scope_names)


cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
runtime_user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {runtime_user_pool_id}")

utils.get_or_create_resource_server(
    cognito, runtime_user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES
)
print("Resource server ensured.")

runtime_client_id, runtime_client_secret = utils.get_or_create_m2m_client(
    cognito, runtime_user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID, scope_names
)

print(f"Client ID: {runtime_client_id}")

# Get discovery URL
runtime_cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{runtime_user_pool_id}/.well-known/openid-configuration"
print(runtime_cognito_discovery_url)

### Step 3.2: Write the initial MCP server (`getOrder` + `updateOrder`)

The sync demos in Steps 5 and 6 work by *adding* tools to this server and watching when the gateway notices. Start small: just two tools.

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def getOrder() -> int:
    """Get an order."""
    return 123


@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update an existing order."""
    return 456


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 3.3: Configure and launch on AgentCore Runtime

`deploy_mcp_server` (from `runtime_deploy.py`) wraps configure + launch + URL derivation into one call.

In [ ]:
from runtime_deploy import deploy_mcp_server

deployed = deploy_mcp_server(
    entrypoint="mcp_server.py",
    agent_name="ac_gateway_sync",
    region=REGION,
    runtime_client_id=runtime_client_id,
    runtime_discovery_url=runtime_cognito_discovery_url,
)
agentcore_runtime = deployed.runtime
mcp_arn = deployed.agent_arn
mcp_id = deployed.agent_id
mcp_url = deployed.agent_url

## Step 4: Wire the MCP Server in as a Gateway Target

### Step 4.1: Configure outbound auth (OAuth2 credential provider)

The gateway needs an OAuth2 credential provider to call the runtime's MCP server with a Cognito-issued bearer token.

In [ ]:
identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name="ac-gateway-mcp-server-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": runtime_cognito_discovery_url},
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### Step 4.2: Create the Gateway Target

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### Step 4.3: Verify the Gateway Target is READY

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(list_targets_response)

### Step 4.4: Get an inbound access token

In [ ]:
token_response = utils.get_token(
    gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
)
token = token_response["access_token"]
print("Token (truncated):", token[:60], "...")

### Step 4.5: Set up the `GatewayMCPClient` helper

`gateway_mcp_client.GatewayMCPClient` (defined alongside the notebook) wraps the bearer-token + `MCP-Protocol-Version` + JSON-RPC plumbing so the demo cells can call `mcp.list_tools()` etc. as one-liners. Created once here, reused throughout the rest of the workshop.

In [ ]:
import json

from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(
        gw_user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION
    )["access_token"]


mcp = GatewayMCPClient(gatewayURL, _get_inbound_token)
print("GatewayMCPClient ready.")

## Step 5: Explicit synchronization with `SynchronizeGatewayTargets`

### Step 5.1: Background

`SynchronizeGatewayTargets` is a **control-plane operation that refills the catalog cache for `listingMode='DEFAULT'` targets** — Gateway opens a session with the MCP server, retrieves and processes its catalog (tools, prompts, resources, resource templates), prefixes tool/prompt names with the target name to prevent collisions, and updates its persistent index.

Because this populates a *cache*, it only matters for DEFAULT-mode targets. DYNAMIC-mode targets never read from the cache, so calling `SynchronizeGatewayTargets` on them is unnecessary.

Below we update `mcp_server_updated.py` to add a new tool (`cancelOrder`), redeploy, observe that the gateway's tool list still doesn't include it (the cache is stale), then call `SynchronizeGatewayTargets` and watch the new tool appear.

![Diagram](images/mcp-server-target-explicit-sync.png)

### Step 5.2: Update the MCP server (add `cancelOrder`)

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 5.3: Re-deploy the runtime

In [ ]:
print("Re-launching the MCP server with the live additions...")
launch_result = agentcore_runtime.launch()
print(f"Agent ARN (unchanged): {launch_result.agent_arn}")

### Step 5.4: List tools through the gateway — still stale

Call `tools/list`. The new `cancelOrder` should NOT appear yet, because the gateway's catalog was last synced before we added the tool.

In [ ]:
print(json.dumps(mcp.list_tools(), indent=2))

### Step 5.5: Call `SynchronizeGatewayTargets`

In [ ]:
sync_response = gateway_client.synchronize_gateway_targets(
    gatewayIdentifier=gatewayID,
    targetIdList=[gatewayTargetID],
)
print(sync_response)

### Step 5.6: List tools again — sync caught the new tool

In [ ]:
sleep(10)
print(json.dumps(mcp.list_tools(), indent=2))

## Step 6: Implicit synchronization with `UpdateGatewayTarget`

### Step 6.1: Background

`CreateGatewayTarget` and `UpdateGatewayTarget` are also **control-plane operations on `listingMode='DEFAULT'` targets**, and they perform the same catalog refill as `SynchronizeGatewayTargets` — just bundled into the same call as the create/update. `CreateGatewayTarget` is the very first cache fill for a new target; `UpdateGatewayTarget` refills it as an automatic side effect of every update. No separate sync call is needed afterwards.

Like explicit sync, this only matters for DEFAULT-mode targets. DYNAMIC targets don't have a cache to fill.

Below we add `deleteOrder` to the MCP server, redeploy, then call `UpdateGatewayTarget` on the existing target. The catalog refresh happens implicitly.

![Diagram](images/mcp-server-target-implicit-sync.png)

### Step 6.2: Update the MCP server (add `deleteOrder`)

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123

@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456

@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """cancel existing order"""
    return 789

@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """delete existing order"""
    return 101
    
if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 6.3: Re-deploy the runtime

In [ ]:
print("Re-launching the MCP server with the live additions...")
launch_result = agentcore_runtime.launch()
print(f"Agent ARN (unchanged): {launch_result.agent_arn}")

### Step 6.4: Call `UpdateGatewayTarget` — implicit sync

In [ ]:
update_gateway_target_response = gateway_client.update_gateway_target(
    gatewayIdentifier=gatewayID,
    targetId=gatewayTargetID,
    name="mcp-server-target",
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
print(update_gateway_target_response)

### Step 6.5: List tools again — the implicit sync caught the new tool

In [ ]:
sleep(10)
print(json.dumps(mcp.list_tools(), indent=2))

## Step 7: Dynamic listing with `listingMode='DYNAMIC'`

### Step 7.1: Background — DEFAULT vs DYNAMIC

By default, AgentCore Gateway *caches* the capabilities (tools, prompts, resources, resource templates) it discovered when the target was created, updated, or last synchronized. With `listingMode='DEFAULT'`, the four MCP list operations are answered from Gateway's catalog **without invoking the upstream MCP server**. Fast and resilient, but stale until the next sync.

With `listingMode='DYNAMIC'`, every list request is forwarded to the upstream MCP server, and no synchronization is ever required.

A few things to note:

- DYNAMIC mode is **not interoperable with semantic search** (`x_amz_bedrock_agentcore_search`) or with outbound three-legged OAuth (3LO).
- DYNAMIC mode applies uniformly across all four primitive types — tools, prompts, resources, and resource templates.

### Step 7.2: Extend the MCP server with prompts, resources, and a resource template

To demonstrate DEFAULT vs DYNAMIC for **all four** list operations, the upstream MCP server needs to expose all four primitive types. Rewrite `mcp_server_updated.py` to add prompts and resources alongside the existing tools, and add a fresh tool `archiveOrder` so the cached/live contrast is visible on the tools axis too.

In [ ]:
%%writefile mcp_server.py
import json

from mcp.server.fastmcp import FastMCP

mcp = FastMCP(host="0.0.0.0", stateless_http=True)


@mcp.tool()
def getOrder() -> int:
    """Get an order"""
    return 123


@mcp.tool()
def updateOrder(orderId: int) -> int:
    """Update existing order"""
    return 456


@mcp.tool()
def cancelOrder(orderId: int) -> int:
    """Cancel existing order"""
    return 789


@mcp.tool()
def deleteOrder(orderId: int) -> int:
    """Delete existing order"""
    return 101


@mcp.tool()
def archiveOrder(orderId: int) -> int:
    """Archive existing order"""
    return 202


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### Step 7.3: Re-deploy Runtime


In [ ]:
print("Re-launching the MCP server with the live additions...")
launch_result = agentcore_runtime.launch()
print(f"Agent ARN (unchanged): {launch_result.agent_arn}")

### Step 7.4: Create a new gateway target with `listingMode='DYNAMIC'`

Rather than mutate the existing `mcp-server-target` (which uses the default `listingMode='DEFAULT'`), create a *separate* target so both modes coexist on the same gateway and can be compared side-by-side. Both targets point at the same upstream MCP server URL, but they will report different capabilities depending on whether they read from a cache or from the live server.

In [ ]:
launch_result

In [ ]:
create_dynamic_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target-dynamic",
    gatewayIdentifier=gatewayID,
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": mcp_url,
                "listingMode": "DYNAMIC",
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
)
dynamicTargetID = create_dynamic_target_response["targetId"]
print(f"Created DYNAMIC target: {dynamicTargetID}")

### Step 7.5: Side-by-side list tools (before any live changes)

Both targets currently point at the same MCP server URL. The DEFAULT target's catalog is whatever was last synced (from Steps 5 and 6 above). The DYNAMIC target was just created and will fetch its capabilities live on every list call.

> **Pagination is per-target, cached-first.** When multiple targets are attached, `tools/list` returns **one target's tools per page**, with a `nextCursor` for the next target. **DEFAULT-mode (cached) targets are paged first**, then DYNAMIC targets — so a single call without a cursor only surfaces the first DEFAULT target's tools. To see the merged catalog across both targets you have to follow `nextCursor` until it's empty. The `mcp.list_all_tools()` helper below does that loop for you (see `gateway_mcp_client.py`).

Expected:

- **First page (`mcp-server-target___`, DEFAULT):** the catalog from the last sync in Steps 5/6 — `getOrder`, `updateOrder`, `cancelOrder`, `deleteOrder`.
- **Second page (`mcp-server-target-dynamic___`, DYNAMIC):** the live MCP server right now — 5 tools including `archiveOrder`.

In [ ]:
mcp.list_tools()

In [ ]:
all_tools = mcp.list_all_tools()
print(f"{len(all_tools)} tools across both targets:")
for t in all_tools:
    print(f"  - {t['name']}")

### Step 7.6: DEFAULT vs DYNAMIC summary

| Aspect | DEFAULT | DYNAMIC |
|---|---|---|
| `tools/list`, `prompts/list`, `resources/list`, `resources/templates/list` | served from Gateway cache | forwarded to MCP server live |
| `tools/call`, `prompts/get`, `resources/read` | live to MCP server | live to MCP server |
| Requires `SynchronizeGatewayTargets` after capability changes | yes | no |
| Compatible with semantic search (`x_amz_bedrock_agentcore_search`) | yes | no |
| Compatible with outbound 3LO OAuth | yes | no |

## Step 8: Clean up

This is the end of the workshop. Uncomment the cell below to tear down everything you created — gateway, OAuth2 credential provider, runtime, both Cognito user pools, and the gateway IAM role.

In [ ]:
## Step 8.1: Delete the Gateway (transitively deletes both targets — DEFAULT and DYNAMIC)
utils.delete_gateway(gateway_client, gatewayID)

## Step 8.2: Delete the OAuth2 credential provider
identity_client.delete_oauth2_credential_provider(name="ac-gateway-mcp-server-identity")

## Step 8.3: Delete the Runtime
agentcore_runtime.destroy(delete_ecr_repo=True)

## Step 8.4: Delete the local files the starter-toolkit generated
# `.bedrock_agentcore.yaml` caches agent config; `Dockerfile` and
# `.dockerignore` are the build context the toolkit pushes to CodeBuild.
# Removing them lets a fresh run of this notebook reconfigure from scratch.
from pathlib import Path

for fname in (".bedrock_agentcore.yaml", "Dockerfile", ".dockerignore"):
    try:
        Path(fname).unlink()
        print(f"✅ {fname} deleted")
    except FileNotFoundError:
        print(f"ℹ️  {fname} not found")

## Step 8.5: Delete the Cognito user pools (gateway + runtime) and the Gateway IAM role
# utils.delete_cognito_user_pool only calls delete_user_pool, which fails if a hosted-UI
# domain exists on the pool — and get_or_create_user_pool does set one up. Tear the
# domain down first, then use the helper.
import boto3

cognito = boto3.client("cognito-idp", region_name=REGION)
for pool_id in (gw_user_pool_id, runtime_user_pool_id):
    pool = cognito.describe_user_pool(UserPoolId=pool_id)["UserPool"]
    if pool.get("Domain"):
        cognito.delete_user_pool_domain(Domain=pool["Domain"], UserPoolId=pool_id)
    utils.delete_cognito_user_pool(pool_id, region=REGION)

# utils.delete_iam_role detaches managed + inline policies and then deletes the role.
utils.delete_iam_role("agentcore-sample-mcpgateway-role")